In [10]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# Charger .env
load_dotenv()
AIRLABS_API_KEY = os.getenv("AIRLABS_API_KEY")

DEPARTURE_AIRPORTS = ["FRA", "MAD", "BCN", "FCO", "ZRH"]
MAX_FLIGHTS = 10


# -----------------------------
# 1) FETCH SCHEDULES (BRUT)
# -----------------------------
def fetch_schedules(dep_airports):
    base_url = "https://airlabs.co/api/v9/schedules"
    flights = []

    for dep in dep_airports:
        params = {
            "dep_iata": dep,
            "api_key": AIRLABS_API_KEY
        }

        r = requests.get(base_url, params=params)
        data = r.json()

        if "response" not in data:
            print(f"⚠️ Aucun résultat pour {dep} — {data}")
            continue

        flights.extend(data["response"])

    return flights


# -----------------------------
# 2) FILTRAGE DES CODESHARE + flight_iata NULL
# -----------------------------
def filter_codeshares(flights):
    filtered = []
    seen_cs = set()

    for f in flights:
        fi = f.get("flight_iata")
        cs = f.get("cs_flight_iata")

        if fi is None:
            continue

        if cs is not None:
            seen_cs.add(cs)
            continue

        if fi in seen_cs:
            continue

        filtered.append(f)

    return filtered


# -----------------------------
# 3) GARDER SEULEMENT LES VOL "LANDED" SI POSSIBLE
# -----------------------------
def filter_landed(flights):
    landed = [f for f in flights if f.get("status") == "landed"]
    return landed if len(landed) > 0 else flights


# -----------------------------
# 4) LIMITER À MAX_FLIGHTS
# -----------------------------
def limit_flights(flights, max_flights):
    return flights[:max_flights]


# -----------------------------
# 5) FETCH DELAYS (toujours null mais même ordre)
# -----------------------------
def fetch_delays_raw(schedules):
    delays = {}
    for s in schedules:
        delays[s["flight_iata"]] = None
    return delays


# -----------------------------
# 6) CALCUL DES FEATURES (corrigé)
# -----------------------------
def parse_dt(dt_str):
    if not dt_str:
        return None
    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")


def compute_delay(a, b):
    if a is None or b is None:
        return None
    delay = (a - b).total_seconds() / 60.0
    return max(delay, 0)  # jamais négatif


def build_features_from_schedule(s):
    dep_time = parse_dt(s.get("dep_time"))
    arr_time = parse_dt(s.get("arr_time"))
    dep_actual = parse_dt(s.get("dep_actual"))
    arr_actual = parse_dt(s.get("arr_actual"))
    dep_estimated = parse_dt(s.get("dep_estimated"))
    arr_estimated = parse_dt(s.get("arr_estimated"))

    scheduled_hour = dep_time.hour if dep_time else None
    day_of_week = dep_time.weekday() if dep_time else None
    month = dep_time.month if dep_time else None
    is_weekend = 1 if day_of_week in [5, 6] else 0 if day_of_week is not None else None

    return {
        "airline_name": s.get("airline_iata"),
        "departure_iata": s.get("dep_iata"),
        "arrival_iata": s.get("arr_iata"),
        "scheduled_hour": scheduled_hour,
        "day_of_week": day_of_week,
        "month": month,
        "is_weekend": is_weekend,

        # Retards réels
        "departure_delay_actual": compute_delay(dep_actual, dep_time),
        "arrival_delay_actual": compute_delay(arr_actual, arr_time),

        # Retards estimés
        "departure_delay_estimated": compute_delay(dep_estimated, dep_time),
        "arrival_delay_estimated": compute_delay(arr_estimated, arr_time),

        "flight_duration_scheduled": s.get("duration"),
    }


# -----------------------------
# 7) PIPELINE COMPLET
# -----------------------------
raw = fetch_schedules(DEPARTURE_AIRPORTS)
filtered = filter_codeshares(raw)
landed = filter_landed(filtered)
limited = limit_flights(landed, MAX_FLIGHTS)

delays_raw = fetch_delays_raw(limited)
features = [build_features_from_schedule(s) for s in limited]


# -----------------------------
# 8) SAUVEGARDE DES 3 JSON
# -----------------------------
with open("airlabs_schedules.json", "w") as f:
    json.dump(limited, f, indent=4)

with open("airlabs_delays.json", "w") as f:
    json.dump(delays_raw, f, indent=4)

with open("airlabs_training_data.json", "w") as f:
    json.dump(features, f, indent=4)

print("Fichiers sauvegardés :")
print("- airlabs_schedules.json")
print("- airlabs_delays.json")
print("- airlabs_training_data.json")


# -----------------------------
# 9) AFFICHAGE DANS LE NOTEBOOK
# -----------------------------
df_raw = pd.DataFrame(limited)
df_delays = pd.DataFrame(list(delays_raw.items()), columns=["flight_iata", "delay_info"])
df_feat = pd.DataFrame(features)

print("=== RAW SCHEDULES ===")
display(df_raw)

print("=== RAW DELAYS ===")
display(df_delays)

print("=== FEATURES ===")
display(df_feat)


Fichiers sauvegardés :
- airlabs_schedules.json
- airlabs_delays.json
- airlabs_training_data.json
=== RAW SCHEDULES ===


,airline_iata,airline_icao,flight_iata,flight_icao,flight_number,dep_iata,dep_icao,dep_terminal,dep_gate,dep_time,...,delayed,dep_delayed,arr_delayed,aircraft_icao,arr_time_ts,dep_time_ts,arr_estimated_ts,dep_estimated_ts,arr_actual_ts,dep_actual_ts
0,LH,DLH,LH120,DLH120,120,FRA,EDDF,1,A24,2026-04-21 18:15,...,NaN,NaN,NaN,None,1776791400,1776788100,1.776791e+09,1.776788e+09,1.776791e+09,1.776788e+09
1,YW,ANE,YW2332,ANE2332,2332,MAD,LEMD,4,K84,2026-04-21 17:55,...,3.0,3.0,NaN,None,1776791700,1776786900,1.776791e+09,1.776787e+09,1.776791e+09,1.776787e+09
2,FR,RYR,FR2061,RYR2061,2061,MAD,LEMD,1,NaN,2026-04-21 18:05,...,NaN,NaN,NaN,None,1776792900,1776787500,1.776792e+09,NaN,1.776792e+09,NaN
3,VY,VLG,VY3916,VLG3916,3916,BCN,LEBL,1,A22,2026-04-21 17:40,...,NaN,1.0,NaN,None,1776789300,1776786000,1.776789e+09,1.776786e+09,1.776789e+09,1.776786e+09
4,VY,VLG,VY1430,VLG1430,1430,BCN,LEBL,1,B39,2026-04-21 17:30,...,3.0,13.0,3.0,None,1776789900,1776785400,1.776790e+09,1.776786e+09,1.776790e+09,1.776786e+09
5,TO,TVF,TO4755,TVF4755,4755,BCN,LEBL,2,S20,2026-04-21 17:40,...,4.0,4.0,NaN,None,1776793200,1776786000,1.776793e+09,1.776786e+09,1.776793e+09,1.776786e+09
6,VY,VLG,VY3718,VLG3718,3718,BCN,LEBL,1,A14,2026-04-21 17:45,...,NaN,1.0,NaN,None,1776789600,1776786300,1.776789e+09,1.776786e+09,1.776789e+09,1.776786e+09
7,VY,VLG,VY3524,VLG3524,3524,BCN,LEBL,1,B40,2026-04-21 18:10,...,5.0,5.0,NaN,None,1776791700,1776787800,1.776791e+09,1.776788e+09,1.776791e+09,1.776788e+09
8,FR,RYR,FR7075,RYR7075,7075,FCO,LIRF,1,A63,2026-04-21 17:45,...,NaN,NaN,NaN,None,1776790500,1776786300,NaN,NaN,NaN,NaN
9,OS,AUA,OS556,AUA556,556,FCO,LIRF,1,A73,2026-04-21 17:45,...,4.0,4.0,NaN,None,1776792000,1776786300,1.776792e+09,1.776787e+09,1.776792e+09,1.776787e+09


=== RAW DELAYS ===


,flight_iata,delay_info
0,LH120,None
1,YW2332,None
2,FR2061,None
3,VY3916,None
4,VY1430,None
5,TO4755,None
6,VY3718,None
7,VY3524,None
8,FR7075,None
9,OS556,None


=== FEATURES ===


,airline_name,departure_iata,arrival_iata,scheduled_hour,day_of_week,month,is_weekend,departure_delay_actual,arrival_delay_actual,departure_delay_estimated,arrival_delay_estimated,flight_duration_scheduled
0,LH,FRA,MUC,18,1,4,0,0.0,0.0,0.0,0.0,55
1,YW,MAD,BJZ,17,1,4,0,3.0,0.0,3.0,0.0,80
2,FR,MAD,PMI,18,1,4,0,NaN,0.0,NaN,0.0,90
3,VY,BCN,PMI,17,1,4,0,1.0,0.0,1.0,0.0,55
4,VY,BCN,BIO,17,1,4,0,13.0,3.0,13.0,3.0,75
5,TO,BCN,ORY,17,1,4,0,4.0,0.0,4.0,0.0,120
6,VY,BCN,MAH,17,1,4,0,1.0,0.0,1.0,0.0,55
7,VY,BCN,IBZ,18,1,4,0,5.0,0.0,5.0,0.0,65
8,FR,FCO,BRI,17,1,4,0,NaN,NaN,NaN,NaN,70
9,OS,FCO,VIE,17,1,4,0,4.0,0.0,4.0,0.0,95
